<a href="https://colab.research.google.com/github/loukikjoshi06-ops/NIRVANA-gpt/blob/main/NIRVANA_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [245]:
import torch
import torch.nn as nn
from torch.nn import functional as F



here dataset is uploaded

In [246]:
with open('TinyStories-small.txt','r', encoding='utf-8')as l:
    text = l.read()

In [247]:
print(text[ :100])

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with


In [248]:
char = sorted(list(set(text)))
print (len(char))

106


In [249]:
stoi = {ch:i for i,ch in enumerate(char)}
itos = {i:ch for i,ch in enumerate(char)}


In [250]:
encode = lambda s:[stoi[c] for c in s]
decode = lambda l:''.join([itos[i] for i in l])

In [251]:
test = "loukik"
print(encode(test))
print(decode(encode(test)))

[71, 74, 80, 70, 68, 70]
loukik


In [252]:
data =torch.tensor(encode(text),dtype = torch.long)

In [253]:
print(data.shape)
print(data[:50])

torch.Size([88018997])
tensor([45, 73, 64,  2, 63, 60, 84, 12,  2, 60,  2, 71, 68, 79, 79, 71, 64,  2,
        66, 68, 77, 71,  2, 73, 60, 72, 64, 63,  2, 42, 68, 71, 84,  2, 65, 74,
        80, 73, 63,  2, 60,  2, 73, 64, 64, 63, 71, 64,  2, 68])


In [254]:
n = int(0.8 * len(data))
train_data = data[ :n]
val_data = data [n: ]

In [255]:
batch_size = 32
block_size = 64
vocab_size = len(char)

In [277]:
batch_size = 32
block_size = 256
max_iters = 10000
eval_interval = 500
learning_rate = 1e-3
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iters = 200

n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.1


In [257]:
def get_batch(split):
    data = train_data if split == 'train' else val_data

    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    x = x.to(device)
    y = y.to(device)

    return x, y

In [258]:
xb,yb = get_batch('train')
print(xb.shape)
print(yb.shape)

torch.Size([32, 256])
torch.Size([32, 256])


In [259]:
import math

# Learning rate decay settings
warmup_iters = 100
lr_decay_iters = max_iters # decay all the way to max_iters
min_lr = 1e-4 # minimum learning rate

def get_lr(it):
    # 1) Linear warmup for warmup_iters steps
    if it < warmup_iters:
        return learning_rate * it / warmup_iters
    # 2) If it > lr_decay_iters, return min learning rate
    if it > lr_decay_iters:
        return min_lr
    # 3) In between, use cosine decay down to min learning rate
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) # coeff ranges 0..1
    return min_lr + coeff * (learning_rate - min_lr)

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [260]:
for b in range(1):
  for t in range(block_size):
    context = xb[b,:t+1]
    target = yb[b,t]


In [261]:
class Head(nn.Module):

  def __init__(self, head_size):
    super().__init__()

    self.key = nn.Linear(n_embd, head_size, bias = False)
    self.query = nn.Linear(n_embd, head_size, bias = False)
    self.value = nn.Linear(n_embd, head_size, bias = False)

    self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))

    self.dropout = nn.Dropout(dropout)

  def forward(self,x):

    B, T, C = x.shape

    k = self.key(x)
    q = self.query(x)

    head_size = k.shape[-1]


    wei = q @ k.transpose(-2, -1)
    wei = wei * head_size**-0.5

    wei = wei.masked_fill(self.tril[:T , :T] == 0, float('-inf'))
    wei = F.softmax(wei, dim = -1)
    wei = self.dropout(wei)
    v = self.value(x)
    out = wei @ v

    return out




In [262]:
class MultiHeadAttention(nn.Module):

  def __init__(self,num_heads,head_size):
    super().__init__()

    self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    self.proj = nn.Linear(head_size * num_heads, n_embd)

    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
     out = torch.cat([h(x) for h in self.heads], dim = -1)
     out = self.proj(out)
     out = self.dropout(out)

     return out

In [263]:
class FeedForward(nn.Module):

  def __init__(self,n_embd):
    super().__init__()

    self.net = nn.Sequential(nn.Linear(n_embd, 4 * n_embd),
                             nn.ReLU(),
                             nn.Linear(4 * n_embd, n_embd),
                             nn.Dropout(dropout),)
  def forward(self,x):
        return self.net(x)

In [264]:
class Block(nn.Module):

  def __init__(self,n_embd,n_head):
    super().__init__()

    head_size = n_embd // n_head

    self.sa = MultiHeadAttention(n_head, head_size)
    self.ffwd = FeedForward(n_embd)
    self.ln1 = nn.LayerNorm(n_embd)
    self.ln2 = nn.LayerNorm(n_embd)

  def forward(self,x):
      x = x + self.sa(self.ln1(x))
      x = x + self.ffwd(self.ln2(x))

      return x

In [265]:
class NIRVANAgpt (nn.Module):
  def __init__(self):
   super().__init__()

   self.token_embedding_table = nn.Embedding (vocab_size,n_embd)

   self.position_embedding_table = nn.Embedding(block_size, n_embd)


   self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])

   self.ln_f = nn.LayerNorm(n_embd)

   self.lm_head = nn.Linear(n_embd, vocab_size)


  def forward(self,idx,target = None):

    B, T = idx.shape

    tok_emb = self.token_embedding_table(idx)

    pos_emb = self.position_embedding_table(torch.arange(T, device=device))


    x = tok_emb + pos_emb

    x = self.blocks(x)

    x = self.ln_f(x)

    logits = self.lm_head(x)

    if target is None:
      loss = None
      return logits, loss

    else:
      B,T,C = logits.shape
      logits = logits.view(B*T,C)
      targets = target.view(B*T)

      loss = F.cross_entropy(logits,targets)
      return logits,loss

  def generate(self,idx,max_new_tokens):
    for _ in range(max_new_tokens):
      idx_cond = idx[:, -block_size:]

      logits, loss = self(idx_cond)

      logits = logits[:,-1,:]

      probs = F.softmax(logits,dim=-1)

      idx_next = torch.multinomial(probs,num_samples=1)

      idx = torch.cat((idx,idx_next),dim=1)

    return idx

In [266]:
import gc

# Free up any unused CUDA memory
if device == 'cuda':
    torch.cuda.empty_cache()
    gc.collect()

model = NIRVANAgpt().to(device)
xb, yb = get_batch('train')

# Run the forward pass without tracking gradients to save memory
with torch.no_grad():
    logits, loss = model(xb, yb)

print("Logit shape : ", logits.shape)
print("Loss : ", loss)

Logit shape :  torch.Size([8192, 106])
Loss :  tensor(4.8046, device='cuda:0')


In [267]:
print(device)

cuda


In [268]:
import os
import gc

# 1. Forcefully release the old model and optimizer from GPU memory if they exist
if 'model' in globals():
    del model
if 'optimizer' in globals():
    del optimizer

# 2. Flush PyTorch's internal memory cache
if device == 'cuda':
    torch.cuda.empty_cache()
    gc.collect()
    print("GPU memory cache successfully cleared!")

# 3. Instantiate the new, resized model
model = NIRVANAgpt().to(device)

checkpoint_path = "NIRVANAgpt_ver3.pth"
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}...")
    try:
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        print("Successfully loaded checkpoint!")
    except RuntimeError as e:
        print("\n[Warning] Could not load checkpoint due to model architecture or size mismatch.")
        print("Starting training from scratch with your updated hyperparameters instead.\n")
else:
    print(f"No checkpoint found at {checkpoint_path}. Starting training from scratch.")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

model.train()

for iter in range(max_iters):

  xb, yb = get_batch('train')

  logits, loss = model(xb, yb)

  optimizer.zero_grad(set_to_none=True)

  loss.backward()

  optimizer.step()

  if iter % 100 == 0:
    print(f"Step {iter}: Loss = {loss.item():.4f}")

GPU memory cache successfully cleared!
Loading checkpoint from NIRVANAgpt_ver2.pth...
Successfully loaded checkpoint!
Step 0: Loss = 0.7434
Step 100: Loss = 0.7622
Step 200: Loss = 0.7406
Step 300: Loss = 0.7335
Step 400: Loss = 0.7198
Step 500: Loss = 0.7327
Step 600: Loss = 0.7578
Step 700: Loss = 0.7749
Step 800: Loss = 0.7710
Step 900: Loss = 0.7462
Step 1000: Loss = 0.7496
Step 1100: Loss = 0.7620
Step 1200: Loss = 0.7123
Step 1300: Loss = 0.7465
Step 1400: Loss = 0.7074
Step 1500: Loss = 0.7198
Step 1600: Loss = 0.7327
Step 1700: Loss = 0.6857
Step 1800: Loss = 0.6880
Step 1900: Loss = 0.7074
Step 2000: Loss = 0.7233
Step 2100: Loss = 0.7560
Step 2200: Loss = 0.7371
Step 2300: Loss = 0.7349
Step 2400: Loss = 0.7500
Step 2500: Loss = 0.7665
Step 2600: Loss = 0.7161
Step 2700: Loss = 0.7261
Step 2800: Loss = 0.6943
Step 2900: Loss = 0.7501
Step 3000: Loss = 0.7324
Step 3100: Loss = 0.7153
Step 3200: Loss = 0.7170
Step 3300: Loss = 0.7385
Step 3400: Loss = 0.7514
Step 3500: Loss = 0

In [280]:
# Optional Optimized Training with Learning Rate Decay & Validation tracking
import os
import gc

if 'model' in globals():
    del model
if 'optimizer' in globals():
    del optimizer
if device == 'cuda':
    torch.cuda.empty_cache()
    gc.collect()

model = NIRVANAgpt().to(device)
checkpoint_path = "NIRVANAgpt_ver4.pth"
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}...")
    try:
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        print("Successfully loaded checkpoint!")
    except Exception:
        print("Could not load checkpoint. Starting fresh.")

# Using AdamW with weight decay to prevent overfitting
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-2)

print("Beginning training with Cosine Learning Rate Decay...")
model.train()

for iter in range(max_iters):
    # Set learning rate for this step
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # Get batch and compute loss
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # Periodically estimate loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        print(f"Step {iter:4d}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f} (lr: {lr:.2e})")

Loading checkpoint from NIRVANAgpt_ver4.pth...
Successfully loaded checkpoint!
Beginning training with Cosine Learning Rate Decay...
Step    0: train loss 0.6209, val loss 0.6334 (lr: 0.00e+00)
Step  500: train loss 0.6548, val loss 0.6650 (lr: 9.85e-04)
Step 1000: train loss 0.6507, val loss 0.6626 (lr: 9.27e-04)
Step 1500: train loss 0.6446, val loss 0.6571 (lr: 8.31e-04)
Step 2000: train loss 0.6401, val loss 0.6471 (lr: 7.05e-04)
Step 2500: train loss 0.6298, val loss 0.6457 (lr: 5.64e-04)
Step 3000: train loss 0.6230, val loss 0.6386 (lr: 4.22e-04)
Step 3500: train loss 0.6121, val loss 0.6297 (lr: 2.93e-04)
Step 4000: train loss 0.6115, val loss 0.6246 (lr: 1.89e-04)
Step 4500: train loss 0.6046, val loss 0.6201 (lr: 1.23e-04)
Step 5000: train loss 0.6047, val loss 0.6155 (lr: 1.00e-04)
Step 5500: train loss 0.6007, val loss 0.6201 (lr: 1.00e-04)
Step 6000: train loss 0.6019, val loss 0.6172 (lr: 1.00e-04)
Step 6500: train loss 0.5978, val loss 0.6139 (lr: 1.00e-04)
Step 7000: tr

In [281]:
model.eval()

context = torch.zeros((1,1),dtype=torch.long,device=device)

print(decode(model.generate(context,max_new_tokens=500)[0].tolist()))

	it picking up a few waters and pladed it go fast as it looked like it was before.
<|endoftext|>

Once upon a time there was a 3 year old girl who loved exploring. She was an old owl who loved to play with all her friends in the wild forest. She loved to explore the forest and sometimes she would play with her friends there. 
One day, she decided to go for a walk. As she walked, she noticed a lovely volcano on the ground. She wanted to see what it was. She asked her friends, the squirrel, for hel


In [282]:
torch.save(model.state_dict(),"NIRVANAgpt_ver5.pth")